# 🏥 Medical VLM Decisive Validation Experiment (Kaggle Dual-T4 GPU) — Revision rev03
**Research Project:** Evaluasi Reliabilitas dan Selective Prediction pada Medical Vision-Language Model di Bawah Pergeseran Bahasa (IndoRad-VQA)  
**Author:** Imam (Undergraduate CS Thesis)  
**Revision:** `rev03` (Publication-Grade Candidate Scoring, Visual Intervention & Statistical Permutation Audit)  
**Environment:** Kaggle Dual Tesla T4 (2x 16GB VRAM, 4-bit NF4 Quantization)  

---

## 📌 Metodologi & Arahan Ilmiah Revisi rev03
1. **Symmetrical Candidate Scoring & Length Normalization:**
   - Memperbaiki bias asimetri panjang token pada kandidat bahasa Indonesia (`Ya` 1 token vs `Tidak` 2-3 token BPE).
   - Menggunakan respons tertutup lengkap dengan token terminasi generasi:
     - **English:** `["YES<|im_end|>", "NO<|im_end|>"]`
     - **Indonesian:** `["YA<|im_end|>", "TIDAK<|im_end|>"]`
   - Menghitung **Length-Normalized Log-Likelihood** (rata-rata log-probabilitas per token target) dan probabilitas relatif stabil via `logsumexp`.
2. **Symmetrical Prompting:**
   - Menghilangkan preamble asimetris bahasa Indonesia agar prompt EN dan ID memiliki struktur sintaksis 1-to-1 yang identik:
     - **EN:** `"Answer only YES or NO.\nQuestion: {q}"`
     - **ID:** `"Jawab hanya dengan YA atau TIDAK.\nPertanyaan: {q}"`
3. **Peniadaan Blur Total:**
   - Hanya dua kondisi visual yang dievaluasi:
     - `P0_ORIGINAL`: Citra radiologi asli tanpa manipulasi.
     - `P1_BLACK_BLANK`: Kanvas hitam murni `RGB (0, 0, 0)` berdimensi dan mode identik dengan citra asli.
4. **Selective Prediction 5 Estimator (RQ3):**
   - $M_1$: Confidence-only baseline ($p_{\text{orig}}$)
   - $M_2$: Visual-only ($\max(0, \Delta_{\text{dir}})$)
   - $M_{\text{dir}}$: Directed Multiplicative ($p_{\text{orig}} \times \max(0, \Delta_{\text{dir}})$)
   - $M_{\text{norm}}$: Normalized Directed Linear ($p_{\text{orig}} \times \frac{1 + \Delta_{\text{dir}}}{2}$)
   - $M_3^{\text{old}}$: Historical Unsigned Baseline ($p_{\text{orig}} \times \frac{1 + |\Delta_{\text{dir}}|}{2}$)
5. **Exact Paired Image-Cluster Permutation Test & Holm-Bonferroni Correction:**
   - Menggantikan *heuristic empirical bootstrap tail shortcuts* dengan **Exact Paired Cluster Permutation Test** ($B = 1000$ *sign-flips* pada tingkat klaster citra) di bawah $H_0: \mathbb{E}[\Delta \text{AURC}] = 0$.
   - Menerapkan koreksi *multiple testing* **Holm-Bonferroni** untuk keluarga 4 estimator yang dibandingkan terhadap $M_1$.
6. **Disiplin Split Data Otoritatif (Zero Leakage):**
   - **Primary Test-Only Subset:** $N = 92$ pasangan QA (35 klaster citra fisik unik). Evaluasi utama bebas dari data dev.
   - **Full Benchmark Sensitivity Set:** $N = 150$ pasangan QA (58 klaster citra fisik unik). Memuat split test (92) + dev (58) untuk uji sensitivitas ukuran sampel.
   - Validasi SHA-256 manifest otoritatif: `988f57a6512223337bb5ab3b192f1802e38bfec2f6b262f51607381faa74bf52`.
7. **Isolasi Revisi rev03:**
   - Seluruh artefak disimpan di `results/decisive_validation/rev03/` tanpa mengubah satu bit pun dari artefak historis.

In [ ]:
# ==============================================================================
# STAGE 2: ENVIRONMENT & HARDWARE AUDIT
# ==============================================================================
import os
import sys
import subprocess
import platform

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

try:
    from IPython.display import display, FileLink
except ImportError:
    display = print
    FileLink = str

print("=" * 70)
print("🔍 SYSTEM & HARDWARE AUDIT")
print("=" * 70)
print(f"OS Platform     : {platform.platform()}")
print(f"Python Version  : {sys.version.split()[0]}")

required_pkgs = ["transformers", "accelerate", "bitsandbytes", "qwen_vl_utils", "torch", "huggingface_hub", "pyarrow"]
missing_pkgs = []
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f"📦 Installing missing packages: {missing_pkgs}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs)
    print("✅ Installation complete.")

# Check native Qwen2.5-VL support in transformers
needs_upgrade = False
try:
    import transformers
    if not hasattr(transformers, "Qwen2_5_VLForConditionalGeneration"):
        needs_upgrade = True
except Exception:
    needs_upgrade = True

if needs_upgrade:
    print("📦 Upgrading transformers to >=4.49.0 for native Qwen2.5-VL support...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers>=4.49.0", "accelerate>=0.26.0", "qwen-vl-utils"])
    import importlib
    importlib.invalidate_caches()
    transformers = importlib.import_module("transformers")
    print("✅ Transformers upgraded and reloaded.")

import torch
import transformers

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version    : {torch.version.cuda}")
    print(f"Device Count    : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"  - GPU {i}: {props.name} ({vram_gb:.2f} GB VRAM, Compute {props.major}.{props.minor})")
else:
    print("⚠️ WARNING: Running on CPU! GPU acceleration strongly recommended.")
print("=" * 70)

In [ ]:
# ==============================================================================
# STAGE 3: EXPERIMENT CONFIGURATION & REVISION rev03 ISOLATION
# ==============================================================================
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import torch

REVISION_ID = "rev03"
SCORING_METHOD = "length_normalized_closed_response_v3"
AUTHORITATIVE_MANIFEST_SHA256 = "988f57a6512223337bb5ab3b192f1802e38bfec2f6b262f51607381faa74bf52"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Isolated Revision rev03 Output Directory
RESULTS_ROOT = Path("results/decisive_validation/rev03")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
PERTURB_DIR = RESULTS_ROOT / "perturbed_images"
PERTURB_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_EN_TEMPLATE = "Answer only YES or NO.\nQuestion: {q}"
PROMPT_ID_TEMPLATE = "Jawab hanya dengan YA atau TIDAK.\nPertanyaan: {q}"

CANDIDATES_EN = ["YES<|im_end|>", "NO<|im_end|>"]
CANDIDATES_ID = ["YA<|im_end|>", "TIDAK<|im_end|>"]

DEFAULT_MODEL_1 = "Qwen/Qwen2.5-VL-7B-Instruct"
DEFAULT_MODEL_2 = "FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL"

EMBEDDED_MANIFEST_B64 = "eNrNfV2P60iS3XsD/R/0ZhuotvP742kwHix2F7bXxu68+KmRn1XsUkk1onRv1/56nyCTEkVK1dW9Y0AP3eQtUUryMDLiRGRE5N/Cz11+6t7Cc6GT/n3bHZ/+dir9sdvvfi67yzk+PZRaDmWXys9h138vh6fjASfbMHzeH8Px1D91/c8fpf95t39KYbffdSlsfz7//HiyC2+lnX7v8vGlnb+U7vnl2P7xts/l6RC+/9y/BKHN03v3a9m2f/z4g7K47PlnHHL59hQOZdMfD6d0PB1Kvwl9v09dOJa8+d4dXzbHl7J563I8hG632e7T8AlOjy9dvxlG+9PTn9/Da3jZhByGX3rFL20+wu55cyyH19AdN7nsnsPu8jvTh9tyDK+b3G2ew1sMB/xu96en//sP//b0l//5D3/+l6e/Hk7lyXCnc8xFyZhiYmm4d8aYsv/1fff8pPF4Ruqnf/3H//4kPY9eC66cqlIW73Q1TMRsbbUqCGYUE5o5afFlaZPNJXnDNGPWKcdKWI0lVFWFuWy1YTxGpTkvIRamYw06F5OzDVVWWZKLKiT54w/aDfeHA2ELiK7w6wDEsavdDMK+JBKAM4gxPHeA6kiIvWz2hM9/nr7/Xwiz6TdeQ99tctiGt+k7vwc97R4RPSuG+8NhkswBvnB47nb9Zl+Hf6ZwyF1Im77bvuxP5Xgsm93+8Ba2MwyPof9p+D9dhWs2v4Td8QShm678l/89R0laEwTnVnhdBPNhQsmKASXOhBr/Rzi5yEJKnNmMew8JX5GWV1FdkapYKZ3USjhVBGcpe+4qrtJcxJqi8Ebo1WjMMi6cCanIIqWvORcdQuA5e2+81EYVGQW+5bWy1ivgJEec5FzKXko4HDdxf8jlcBORPwqCfFAQ1AiCejpCyY4o4H+YCOHXLmw379uwKxMAmBukn7aERZdJ/YTXviMQFhMmeisz7r0mXVLN8oyCaigw/8QFYwMKNgTIda5RQ9yLsaVKLhIrimuhlBbG1FSDKdzawCLzNebquJY2RGmzCKvRQvUp86qZKpVF5ZMOXEXPVBaS21qjjQBZWqWYFlGzH3/gng23SMcLDjA0m7A5dHFTD2HQ7Fda+h3iANV+2hIQh1N/el0KgwhScS65sdpi0uYGA0a5CINTYkCB3l2xvuCZ8d5Cxt0aLnQWtbhqqk6W410LxUyAdNgqvSiSc5OMr1o7tRrM+RBTzk5U67O0gmteopHOGmOChbiloJxMwYgUgzOOUJANhfmUIBB2m1H1xm0p+RqEcsjhEF5wCSwQadolCFnhRh2vuHPctotnEGRTnvrJ+IaBrw54WC2zqtzrLIspWlurIfY5+iQhM0mkXEsQVSmvcKHOSXKVZCpsNZbS0jmRCaUcHDfVFBNtKilqHU0MgCamACyKZbFw/eMPgvHhBul4LQnPh/Lx0/eX7ki6FPrygAeuAxk5duHK+jRcYsmB5tHuCIDoOzA7mxBPP+E/2Jzd9Kf307F7WU0hHUQQsjiGZ/V4yQ023NcIGxfDf8MEElZEhvfqTaoy4o3C6EBWXA3BARuZYWEitAOz3ljJZfVRVw1TxYVRJq3GchBCwGxomgQSSctj4hXQyZpYdgr6p3pTIl4PC1kSbL7B5hewwTIDnbTZH4BEOR0+emjNTdltYY/KG8CbLBJEdv/W7fBh2B+O4YzmkSQMcw2Qbgv4DYArr2B6MNyYgKfxVzv8KiYkgB++fPmxpTQ6zJDKITNZV6jecIbVz/QzMBuArcAVT1gIn+KgkBxEDrNLJF+TdzUGprzVKmlolapFhZJhhuRLW5ETX41WFVQ4IWwdqSdtTSnVJEgyME34Q4xQYUI7TM4A4w5gxWjN6TgDttmpBuNlSk7G6a28xdKHw0qsrMo+Z6IaQgVVz2Ilroy0GzVzZnjRVpUIWUhSZg/lgWfJyRjOkzIRcysGmySkSONP0hfDU8qpKheMX49WnRA2ymSCrsFyWKkgAUbgFtM7Y24n5yWQSGBDBVYLzy9HLkjHmZlOEK1YttvTG7hf2W32jQiORmtJB2dXE1/uXkiahlm6P+4HUj3asZH8XQtMqsxxvK7grIY+LRNg0l3UF3QFwRWhqUw2IqiYZYre5VKhVwIzMNcB8GSnOVih1M6D0UGNe6jiALNkuWRarsZiYH2Ys8rCfGXQRlyj8e1gEoCtFRpOG+1hDqHtnVEVcKnRkNHxeh7G/a6MfBl8cP8+zbq189EMGrR5Y8QD6xl9ixU4OYhM5tlwVnic7DxGH8Fh4gmPO4ADjS+rgjmysDkV6rsUCeOMaQCZwwzhFqopZC+DNy7zYFQkG4HLDM0iuRoL1h0aHShmV4SsxJthGzy4kKE5iFMdrDMV0LAE5ABOY8aiUeMzNmm/6/dbMBqSmgmkLeZRf4XM63QZVM8oPuFw+on+t8RFgcFEq3gyMqlQ9ITLnApPhj/hl3gEu1cgcCDznGcF0+1B5AKvhcx8shWTDZIkjbAGJpKDwukQq4eorQYD+ckxVWd0yQw+iLQwbfi+J+0NVIzGe7GgDj6D/AQOYFzT3m6pvaGE/naCmQIo/Xt3WNu5dHo9vW+glJ/7QDNppXCY9hIkrXibMTSrExRupnAdG50nWCeNieNcFqVAVXDYbxWtVWSm8XdoDV08I10hoIErtBbXYCQwgEblkFaDMReiAeGDBjIKMlPAocCmwIbAjRT0ui6QwRgD4edcARR+fFl0XDLBuN3v8yZt98crCJ5Pb+/gxCBARIRWABSYBFHIMoDdQhYnALweAFBCPkF7js8PogZdaaAZYzYgtE56k1UuQltw1SqgaKEJoHJV0Hh7Thon4B1CxIuE17kaC78BNWxVKhFTJYoCVQNq7XiAayrL4JBqcEQnS7GOwR+Qjf/Ia/5z9hcxVY770+Qa0VzpX8L7RYNgjlDg4tpJmjxsWPvT6+4jrB1skeFM8+JUIC/HNYxkIzvNKrPRKitfOLwdPLkwpEYxt7MPHiq1iGAznGEXc2bOhQIlakFkIBNgfPiKMiWw1WjRe24iLwXOFuyuji6BT8XKYIZAs4PhUCTwtVnwwlaySpLJhpK8Quk97NKhQF2WXMDu9qeLAsFHr8NHbxSUABzQIWSkd8+v+PzGvAkMHn3SDPJvIchnSOSZ/0k3ig2XeKvC43qXMB8MpINByHNyQkt4TtJI+EHWVAXHAWqDIMOzgQXDtoCRrMYStsjAHCQKVhjKB3DEEhKsEtRIBU/2AladzDyXMQoBQMToR9LxrFpHP3KwMJO5OVB8DX/Nm22pR1jjt/f9AQKy3YOu/GnuZ7ZgFj449ZfrgNQw0Xab1+7w5UAN7uoRAzVSjvaIjnMpegs9JGgHnyp1u+czKvTnQASmL/F0JDk67X45vQISSNKuI0E6kiAtfFARhdYlgGzhxcazHEkxc79GSJQ0hsGldpgDkAU4B5lV4rGVecGSZGAjMM0OwgT3wksQW9A+OAHWgiYHvxoLvM6qykJI2eVY4JJpDeeevNQMexYAPTl0PkbjdCpQv6BEIyRqaYlI8RxCf9yc+inwRz7D/tstDvM6XjyGSHP3fNrRHGtqqElWHvyHfiVDHARFZF0sXB7F2WS7cUcXXWTZSGoEr14ocjUHjSLB7h0cLIFpAr3uvY8ONizDcfXgg9XCbhlnoVAUtDBm6GowqCsQGiGSFbUK7p0HcYyMQfdwGCr4c4oLyCS5/g60Eojp8Q7p+AliZ3guaDSQlgJDnqC2OsGhtEKKKWiB3x+eXxKn427kdBZvtTj4zyBaEcYWtw0nMRYYXK1KhdcDxhJxhBwJKaGgaoJqpLBMhre5Gstyg5kHpldsjZZkKWP2QJnjkSW5rMYl/GaFJw8rRvYa03h8fN/s1RTvTPseNPcF7jmcT4jBtvRkqg7vZyT6U8Y0urruGH4Jb1+2Tv5BrRM017g2wcRCJHb7TflGse80aOQIjnuE4jgHhdeYXdxw8N7XYXLF0+txiAPCkl/ZsDmilX7k9atQ4k4fFEoxihcdya7lfRl19Np8bcL7O5zyZdB4acBgtiBmYIyvdyLIdxxO3MEjOpyqxSfoeB0zBET93J1aAXalseliUsk34FoJ0V2ExGMiZBpC5poabfrw/Nwd8ZRjcHUZtZhF289ximEpCxdSRBFf747HtQDlykPWEfKsYL9rPsNjzvBoM84wKO0sojHViYT7rtyD9kFda+Ek5iQsfnaeHAYoZA2fPRRynmwCbwIFyGw1FsuSeQ36CPoDwpOrq0VXZmEobeUwhhZEycAXgUWEP0/rqKItpAp7D55vXfk+qagBqlsQ0ZzqyAsbYBl8jsOdgPQ9fOxD4tMcUjU5pE0FAaI0mPZN9+1i6seQ8kz9/OWvmz6Ny5qD57Ew/vg7Dt/KLixRuhN/Vs1tfbD4s/KuoeSWbvvu44Zdu5i+90Pp4ZcugvdvEd7r66m/NmrtJ5ZQxeilixRZgbOtLkbNuyujZkdnTYHrOmMZ59BCIcCJU3jnePe6Vs+r8JbjqV1JIIOZYGVVZimrhUhoHVejeags8Gu4IBAZuPsgoYnz5DMrFgehk7EUUzM2Wwe6TWD5BtaSZI9gjOTgN4BbATY49zfxapkKp7f30/bL2PnHxE7zUV3R8XrFFOZ8d56H6XilozAL4Zq9hwvnJpP2T//8j//087/+87/9j0+JN0Z6ROKtm9+hZ37HEoNBY9fugH+Qj9YN+Sr4R6XFIAhK/w6TB2/u1uLiErKznof2g7ubOvxx8HwP0PxhcyLpm69GDpay+X/pdOieQ/e7UH9Md4cM04C6UVfuTlN0IY6csjt25O7QykijX/070L0O3b6WLWAPs0Siy8LItgNDXc7VO54x7uUhPWPdYq10nGXCEFQNJsjmLkMq+/P60Y3lkPJ2okWQ6SsDPJ9kWt1Z7sZNPOJyt27uMx3n7KsxUmIOC38Gqh6ghW6aYqTebjsy95Dw/DGRMA0J02YW5hN0THopRCbhNo+rQ7f5wu6DJsxA0fGd1xKuJ9UXrR7GflCr55vV8/YGvZqzhjVi95EaptII15cBso8JkGm0gI6zZWrKGflvVzkjLbPse9lur/JHZpZvTDP7WtrIRW3/Am3efzUgbvhD5n0aMS6r0HGRRTKLWr3AzTvMM0iuJGyMTr23/AfAhcuX0nVnTRLDPuSaJGX6jbAsQ7zduNwECrUdsj7C4XUwZzsKRA1pIt329LacgdNSAdElCuOd5+J5PZsiMbngoukHgN+CN93FUD8ohq5h6K7SYl+7vCsf/dLKgav+cl63BAPNXX8xc8sFXpkSVFM2ifsiZDwj4Vae8hBbdMzTvaekKPkQ+sZgjsmCR4XrqyOTEerKBBVczFEaOEPaASLhpVyNJYzw3lXmmas2YBZSTlGK0jGazM7wLGSSUoCFwcV2nIDwDQg/B6Kf8n832/2ehGhCZgHJTw0ZMvtn3fN7gfEPCYxpEmKWIYSetPUsTE5avenztRc8Kp1hOv0SDt0wua6+sFxu4q4YEeAqgPxqM6XQ4C5GkKR6MsaM6cJwOxzILayUjcJmWCIS+0K5NDw4WT3Ij1SMp1xq9Jwl65OvQUlGq3duNZbUpnKhlTQ+FZWsdF7GyI2iBV6XMrRyKlzBk7GWW50BUuOLZsEXwzkh4IpAzxZyBz50myhybj1TUcIVwCTOk7E3M6JotR8z0uA/WFVDgnchRC6aNIuh4BpUhi/a2SSiSomUBEV+XVJJC4Fn0cnLIlZjKdCDXMAkLIy+hOCYAIFgFT4GjH4R+MlSKVZXaSXdQk5sW12h48zU32fML6Hvtive/LugsEw8JBSyZY9LtQz+t0wIyvx878sp71P/0S9Y8/D3164/hnN2xDqdninMbxgFkwX0woSHHF1OEJAnK8Zkxaor55TgbnzkcCBAUZLiQC7h8ZSgGCUc66hDghIR8CRzgjUKIIPGZWia1Vg1c61ycppyqlwBs9E111hKYiJLTJGQcizJBimtDDUSHqbhYVZ4jCB8UChkd5UrcheT3P1+WMwjwgLJGm6QjldrtEOK3SZtSzjM8mQmDrKkttcQ3ElkxRiPmcjq2lui4yJyM8JQD2Wgt1fJiIuCgwma0bkcVmTnaSDgdNdpijcjXneRE4+KnGzILb2CCuHoqQAwrwPSOwo3Eyjb8rZeA7pToIIhHrNAxSnVMLgU6lw9/c24xCAPP41SQfanHyKjvw+QB63YcXpUtXSc507FPcgaVGd/LLs8c6pP/bCs3DTK2zlr4as4aPOoOLiGw4W1zjHo78eszmSVsPkyDu5BcXBNSTh5vS5ze9F4XBf+GhXHTz4iFfd8nAF0XCycH0/54zcXzKEiCy0UjHWyR7Iot1bL7zhxGPURnTjf4nB+vTyXjvedk7/8dU3Eb8qCnwJnDyYLLf2WjvcDZ2NK6f2Q2SyHdOnJ6+ykB+nD69C2nIXgnFdLGTejLsDtR8eCDngpXjrcJvwKKz1UgdFgjkJR1TkQGcoQFfNZSS1kdjwKyqxYjRW8SEKwUJzmkAvDFfHNUuAHOfJQPPELzrOPrnowVUJDNzSu4mVTQtI6s3ZIPvrtxNqvgqIfE5SmLoRZFb+M67XP3bfZCmHuYjl0V3miyzQszEsfq3dwJuGt1/PzrzUDHFR6DiNx7wl3m6D8gyP+mCLVL1SXfWHa+1KLFHA54anaYKvR0fvKzGosIKFq9pXH4r3RTrBkZPERLgllqgJCaJ3AHTRGphpaev6mGVZ5RlAM5Lbf0pGj695yaL4a3/JTItGDqUY5LofS8Xo5VKxifaQVRIv4ffm9S/aQ713y9th88dgtYgVHAZJO+nGJwi+nt+058EnJrnfinHcB4Y8JSJsI0q6rDl72b/tnEIN55cqi7qBdsiKMTKVaLIYvsFItTYFAmK3YmankLUNTwWUURYDH6Qq/UkQDu8ioXADGzkcZtdQ8wKAWZTUvWlRMBEdZZsatBuM1Jfx4yfA2RYoGnqOHe4mDK8wUzIMgtC8yw6+FF6qBgm6zQa8r/vvX03Z7rvnfvHfDMd8v/qc+I6/7w5Rwcp1Kcbty8h6/0Owh+UW7QTrOkxDLZuieM3HNVpT8ayolb972w0yjFbiwrdMlxxfg9OvM8hL9ulQt49/dS7fpS2vdMiRxjhdlgLlCETcuK5e5eo2HmTJScJsDisI5/DcuEuNZYgZSxajItYWWrZTk6yEwHJbVkm1VzhkZTTIxBBWrgZLV8DBCqTmuxipKUZhVQllDR7OqjPXQ6JiSRnIugjIw3hEKHpKbuKQuE6w5rcPJUuqG+ddig2OZ1HrxbpyKuRs/Gquhbi3O3caFhn1QXOyEi/2EvzaMGo0Lx0+huqpSGAjeAN72P4KefUz03ISes4tqxGMHrv+f+mbZRsIzLJuPSB73FwiXS5/voD6FktKHKEGTvPGcCFLuho/IPjYcF0YxKChtBo0uJGiKtROI7kKOVIsxQoGl4oxQJvuQuNGyVJcCWUqYOhaSgW5LTFpXoLkyc5RhDMrMc7FkHFZjGatgLh0TnBcYFR450JKsSl3xqrRyWlfrQMJTBoVOlkD0qoHo1ZkntDWdS1+v/qWrlLAY0mHfn9tSbbvdxas89+4aVnmOA0LgF8+lB8mAgut2Q+uKZyi2vjWoWmq120kbdGePmLTBeSsZHk5mK2Ln+o/abY9UOn7X4wI+fUfhzMu1dRXvB/2pZL4MbL3PrYyaxmx5mvqJSggGkqVd9CVqT8lfcKJkKCpzyEjW3HvwInBpvHVhFWUgauJPIsD+OctAOUJcjSWr1hk/FeFuOQ4aVktKFl+JkFGwiiDB1oNxoDBF8UGepvTI4WSxFtREZpSmc+0V9ag4DBOP+tu9fLyFZUOZQYbGDnDD96ecjbJ77agPxvgDX5Qm3NdjShNvTYjoZJ6pMU5GuKjHQ5eomu9b13dxe5l540evZTtNvdu5d/cEicvHFKS2bjScXEczIDxNsYdah0zqpRIfq83K4Z30zOG00jR3lDRvS0GPpqS5nbCYd/AYnPihDeQ3UihfqqNqipg00pcy5mjEh5wueB8jJHQyU779cf8W0svme4ArA4zSK7y6eRVwWzUqhzHiBYq9vblaeHupmYZ7xKVm3Jed8LDXereBMM/jCT3pkgGj5eoh4THl8UxgbcNQFrQSmbsQ2ceESDd7TSefdP0bw+U3ZlELmg/YfN4H0FGvOjiUEHbjINoTMroZbeGfMMFb7xMG9QIKbRz8/GJtCR6PrlilHlvRaAPPFI8G/WF00rwKG0PC8weRTA1yNZagxWpRHFcGQEHBJBOAHmaptpxa3jmALhlxayq6EIRMqxgYThbLB/2xLbSmXw83MDnsd8fnshu81JY2+L7vKWsQ5CYPVaxf6Q1IIz9mb0Dyeho27trLOG3f9rtw+NhQhk8BEXnZf79o374b/ghFswvwKYbVpskwL6TlLiTuMSGRE8ejk6vEljFFud9TyXPX96dCzXv7VpXz8fZWSO9clPHw2dBatCUMbqlKc/JUqf3q8I3VIu3tanq6ncespie4G2LcXCN2G6t57fjotJe/nbpvYUt9A8k16/69TF7sp79wK0rwG7C/jeypp+qy1nr5t7/R2tPQ9D+9ng634wv3Xhs3j/raWqMavuxUk16onO/XQ/jYnHYZVvZXKL0ZzajgW9eqcQBlCJG+Qx0cvqoDpnY1j6cDpJygWSQejH2PbsJyKR9b4rFJ4SXcaJB1Fxb5oLCoSTWqhfs7FJzDGnRg4lQn9FLe9ofDC2HV0lTuNEaf9QE9F3kuf+y3ivMqiIPg0miNqQIXZIJRrYvagxO0EgdHx0VeS+WYWTywSguuWitZPfc6CIqj1wI2omoCeXdgMZ7VqPxqLJUrnMdCnmI1RnllnVMgJkFkuH7gcSEIqTC3uSlZ0pItvjypS2OWxf/DpCNTuxlxOMWhqXyL+Z1RO4Tc7Z8PoXZTC4DL2vYY4WtLFtvBKxgs9DNcqO3gGy0K+hjRyJAVd/B5U5zQM1d6i49MN3FKCKIMY5c13BdhXEyAxispKuPBQULByBS4LpxqCBowoJanUoPPAfLVaKHCfS7GFFVSLLkGFxJzpnpJnmcFplkLwaIx8ErhMRJ+btJbTlxPzhG90/Mz7SzwbYwu7w+UrZxhQRYO5LgK/AmSLcqMr5ZXaks79q/9GnhOPCh4qrX0HU7mSp/63IAK/zqrLz6ru8PpfbFWNl18WRQraUjzXczUBbNJwlID0Ci81dK1JVW6lRsmMvlQPFSZySzxaDKH9XMGHnXUysMhh55TWsCOcp+h36ACYUUBApz0Ikriq9FKKUJxmXNkMkSBCV/hwIdCjTeZCI66s1uuYX4l9KvmhBZvCo9OvopWt6tgCp+idclU/E3ldg8zrh8WMzthZufdlc74YELBE/1oval2lKbWX7WLnuWrzGzqfGqSBzYEOW4GN+7YA8XtQ9oD1YrYhpO5kJ1b5QyO+5/u9+tvzXFW/WqN4SlQXYhWNko/4dBK2DxcPivGJgjKFXpAOOvRJgXyGHQOkXOHdw5qqUrQ3FCzXx4r93j72tuqIStCJQbOsRoLcuEZtdBJRhQfUs3goY5HSdFDG3jhXrFYYjUsenAVwqGlsgwnV+xivlz4viunt/1iwXmW73b5+HXlWeUsYynaWtBp4c1ZMCS72sZhIlxgy5EZZzwUrZJOaed9ztpACRtF8WNhROA6+iiq5r4YbTApnJDGVidXo6miVJGFh8JyDBFkzEqXUxKh8qCt9CB1mQKJEEUNuRwg4RMkfFV7cg+IPwYBf1QIWp3AcDJfOTh3eVgUZrXWDp90MLgDgZIPC4GaIFB/v/q8uzCoR4VhinAqzT/JJ0hhm7rapXWlzVXywGvY9m2roK81V6VhH7G5Ku5LTLjc6Djb4JjRkXXH2da5+I9AIh4TEjNBYsTkYm0+9ieia5tciI6RSwoWUijkM8WT9il174PFhfK87Kq0pdYM1LHuvEp5xvLPVIRDy07E6rbhveunzjt9GIqLKUmg5UGNTK87nPDDcCjeL/3sLmucbagvthunx3vEduO4r0lpm2V1198T/XPw4Ar5JeR9OJ7GgB91RqKw3uv+7/sS5GO+BD/py0VZ90jDIwX5juF13Gej/LraCGDGvd+bMIeOwqL4OzznPpAjE3pqOQU2evoeDn34qpHxj0o39JT2oqdW+TPNUbenLl+Fr7enWfrLlTJoZiaF7jDuonSp87zZ/jTD87IMTgTPUZcpdZRuYyTrTF+AclJSx1Ir4UKoGEuC1JSkdPVMWVty4TLkpKUOMcmQZLGFKUpGiEFYqilbDga/UHjq66g1PCKBT4Wr3EkKLEuoYZkEvipYwF9iHvbW0q1gfjhZFHB+DtOwwcR/GBbxoLDwCRYuLh2nrup13rt8IoW2P2z6UxzPX4aNBRbJQVd8Zbri8v1hn6TpF1YFXrd1Fe7qIXWVnvKD6GRRSH0TsKGtIG3JOTRdurd52W2sNr+cnsOXAZMPCpiaAPusOOwSh79ZGzYB9UW6h7Eeku7pKUynF2G6sfphwmC2OLuQk5Gxdc+7gf5+HQ79mHCoyYYpvlxgGG3/sMIw7GbyW5G2pn3atQvlrLRllL7kA8ZmOUy4tG4NlGQ2JZpxPKEAJs6EUKnUKdvMWOaYMzXij1pKWu5UtThjpCja4PmFTK74IKpwq7GiVIpCbgRVyHh0k70NmK6miBhdTFzbXKHyk6iS1cG2T0l3Wi1N1jUWJB2/84nFYz6xbaFFOrku8ZgJwlXm6lIgrusmz9mq15nOX0TJugdFyU8oLVsn34Zmkbn7x7DwD4mFmWidYctkzPewqd327VYcmv6++T9/XpcL1HGzNstoSSpOEWhztTEEbmLs6l90BItSEg8I9qV85DKLikeCdfQxKzyXxnNaoZQCD1NCOY3f1cBJpRpWo/HiE094dM9lFDyD/MiauIF7AQQDNVqyDkacgwg6WyiLzrR62uHkupEa2ZJCO/ik/ds7zoe+x7RkcbWv5RBc+WkMsYy6dNtRT8fby4fGOZBU2vQOjk1xE7s1Yrb2avloWaKHaFgbMiUQ1mq1SdQwQYlsXKwiwAgwlYRmvnhVvC5VaBOiZTJKcFe5GszrUCyDQfIiWfxqih5s2cMl8lnwAlbrjAsyJuOl83WEx03wXDfc23bfymGC6Pmw/07ohMP+tMv3QBrya+BxUgvxeHoZa09e4Y8ffidW7kGxmqK2Rl0xtA2QOlJOfzkXI15ic630kEpwXsMQwDiuK0lu789I4zzi/oxUgTgBsd59bwJj+4XdmCdUvoyHfkw89CQYet5gjXbC/dZdwl6zQoDcba96ybYr5+UAl61h15lpeCrpIeKK6o/FeR0Qw6+LeCvtCSdCBB9VGZw12WTIENUcmIseBkgbeCRWSW6ztEFRyhquicyQ72JXg4liMDuKg23CtfC2gYyTILtJalaT0VQEm43UmIlRFT0ANAmM1telR4DkUGgz4YbA/y9g9IMCY9oKKZ3caBQ/TaEhQZL+cN6fL/b3p1ULkpIdH0Kln+19fifJEffzoEmOcNIbZHbWpm6GzaWcklC6oZLvIDMlWs02frylq+8BZvnDAuYnwPzal+5Ph8Gof0J+rgoAx0KLzw37PfXdGPLDqe+pnMBclRM0LC78kHbA+Dgjd5l+w1Wz6q3NfluW0H0VIveQOzLzqZnqcLLu4wbXqe4PV4q7NXKjmTbfSXW4br0aeQeOcz/Vh4NDTnAsV8Mo5jRktZwhGTG62/BvAGoIVw0Zd19GRj4mMq04cTiZ+xOhO3wPH0MNV38pdJvWFGLoZ6uAUyEXFDWlXW+GipSp+m2243kO/Y3U9NubetMNPeSm3rgxN0HmljnYU9v4GXo3CuRiNy2pXuM4S8WeCuWW0M7qfQYZfJsaaQ2t5Qnfy69/HWn3oEhPecZ0sqhLFZvvhYpPSCZXuTfXBXWtyiTu4/64+av4ammqdY9ZmmqnvhR0cnsDnqFG5+amRF+WCfeos2+ql6OTJhMgkxALzIbvh/2UzXBz06EpL+RWovRXcXnYueInXPxnuJT/GCwyRJYV6LDEzVRmz7CMVNFb8+TbVlWJa0F1MQycWKsc4bopWChYNY7JUinXvoaaAkgD00nB7knBFGQd9JmqSFdjueisLDCSJfqM6zhVwlvhSlZCwtxpRX3chA0SY+khFdr6RhXp5Dp9czebLR9fmyv3Ht6zh3z48cmvo4SnS75m21h33j5k1iT8s610KToMTuG1UtSfemqTzthl1Uk3HLyrGa5UrHBuhAyS4QF58kVU6lVN28ZyRcFBOEDKVJiiYiDrvhB+Rda4GsuYlKLiFrrQM++V9FJlTIeigWdI2VjcBlVcRPhZPIcffxjnhb9VRdZmxKzJwXWi92xJbnKlhhrElvV9/t4NW6uHPP9Em6c4Jlv1PGHkZyuWI0bWFZUTUyJylWzyFe4i3ErHwQ81A/8ThvogSUDAbBVRUFAUDqUvWUKW5GqsYFXJ2cnkRK5JxmCN0obkz2WlLWW9U6p8sLXQzqv2xx9aE0dhV4Xrb5QSsjat/+tf/3m1+4/hSbCaC3fCQJCnJxbzvaWEGg0pVCe3YpD/XAUXHjZRwvLFYoNWvlINDVxnr0QqOuItU/66yToYFkKUZjVaAiEWIjlaaKF9ia1g3pQcNAekgAAD+5BY0K5KCBkMaYukL1L+24I1CBR14/ukX9+5uc754rW+MEHbaCHrtMah9RkRd6M2Cbaf1UjLQyxC5EMptGsfTfkIWSgFrgB1QbGYDazqIErWMAVja4+Ywno0J130gJF5Eak2vzrvJHWwyzEkU2v2mElWqhTgYyRdgcg4VYT/HBGaNtRNf2iEnT7up7W+ddvutRsa57em2ef8wG9QPOdG+l+BzD8mZG2JYbbAAOo/tdA5e0vkUF4E6URJD5dSpIG4D5XNQ7/wZWmlzo5xQ7VkKnh/BmRaapiV0oBvMJOhFmsi6ugg8clp6lFYnOTeFxOopxopGctjqQY6A/MmGmULtX1YjSU9KIeq1sAShZwKfh/WpgYmeBFK0myyujCrWBAhWOjattCwXmboDldgXNnbEY8/CIN+RBhM21jsOhem7eHXtvCbrzmNWxte4i7DFTdmB4xEgbGHFs8+czVtMcGMvlrNHfdoJ40HP79ymADcJwAoAc6G1qkqS4lO1BY1GGNzCNJKmjDOOS8q9C+MbVyNBgtdKiXv2Uj7+CkeRB0qkuh7oCeVZ9qb1GQfWWWVdgczbeunCw5/eOfHr4NhHhMM3pqJ86tm4n90Yf/O03OhHvTpp4R0I+crStOcWLb7a1PiEsC+k9yRoOUFnCaG5wLnmXoE85YRbqx8cq3rWBm6+lgB3ZCoD6512sOn0tQpF3goRRWG4J5Q+8aUSBvwApYIllWcANFajWWUt3DYS8C/E3P4ILlaRaE2W9A5UYD9U0hPBOZdCmoIRE5xyKt4xvthnwqs7Cx0MzHyw74fVpO6C+G4H+CpFixYJO6p/xnPU+dsDLfSksYIzjScVg03Unq8WVOcovJoXy0Ydq3MwDPxVSaYy6RgXRWYFyRIV5AssRrLK2hDaWryHD6uMxx6WGgYTl6ZY1C1iTZ21jJrmZKqhtBoE8IuEyjJKfnpXlv9+U6LqaOO8+usF9Beiyek1UHPsj7jME4OqfiTanWXUuNGB9eaQaP7FFjigvb7KlWKOGzHo122YOgRfooJVoNtWHBrnTWtCq3GUglf98boKKgo3Emm4ecFTs3JLXOxKEhnAWqBS0giOWqurba6a3sx7i290o/DHtKboWvv8vVnGUqMAi4VULbTrqT43dEvM/4Js3v0T2kHV4XH1IY6qAcoDgGzSDWlmf4zAe4GN6DgiasIYpU5dw7/L4YpHvlqLLxcxwRcl+x4SsFbO9Th4oFpLkA5YFKUZF3QoF7WU6aTb9E9LxZhrFxub7951XTmBO/1MF5Qu7LNN6Ooz0MbkLYl53HY2Hy+F2eLSw8bw87S4G82XnWOwjqY0konB7GYwPVXHVRGYhoStKyFwgW8oWZKQg4O0NM2AoKxgEknaFtOCV8Y7IM7Yy3gZ7iYC13qajAB963ATxRJZKZqAP+E2kpVwmlmmKqugMEahRekQVhSInRblPB6a2EQ98P7aShK2+770xsw/bbffpuRkeUFA0+NN9pKRaczDIvVYMbKtb7fBMjM39O2FQYkl2kbUTxMgF2hfJ/KQ4SasJr6O0bDTVYpV0/JdrkwCYAEtegXwYHerAaDiCb4UbSXtYKC1yw64W0WxmAYFVlS0UIiPXwBSvKlLoe+Bcj8MsVwLNed2p0s93VdfbjwenkK0SnYFanBJMMZhRYQExpmWIxOL4PvDk+32FRp6biapHl0hbOoYJOtsphUtKc7rJNQlLPAtfFZwC1RAkqFrwbzmGzK+ZyBGTUgN4raXvpkufWUuqclJcgrKCOoLacw6QQfBZeOy0ygc2f2Ns8uaVL9co/7dfrCMm/q0326S8F7BheBbQAUUzo3taychwrc2Kc9sRKV4TZy7ywrlUkJXQK2DTfMquSDgXqGS69gdnKoUN/ZEyvXDPrV+rgajUdhoPElkCqYqaqQoPgKm8eql1Uyi2lo4Tv4KhJABWotKZGO16gNVRCzErrrbYEWSe65u7kV0J1QkZg2/HiwUFFLnhJaXYUW0zZ8GxvIxsP+dSYvTTLG3jev4XaWBk1yipRKR7oxTSFF0TKkoCeevPWtUwIeWkgfQ9QKBojDYlGyBtisMi7GDGPuuBPOJvCXgseKxGKTSfD/VVCrsZSVYDQmFvwBWpmBAmUHbhMdU5AEVSOYHwwh0IU91B4YtJgnHdd1fVNc+VYY8fLZQptCUKUj/qxhRcNZmwoj1g6ucElYFwWFvHQidSdZoLiwhmjAIDvlJY+xgmpYZiKsgtWOAq6FtgYUq7GSKcOKM6g6+FqB8yuHNrlwEYSk1gQqB4iLZpCYajP78Yf/BxzgFts="

print(f"📁 Revision rev03 Output Directory: {RESULTS_ROOT.resolve()}")
print(f"🔒 Authoritative Manifest SHA256  : {AUTHORITATIVE_MANIFEST_SHA256}")
print(f"⚙️ Scoring Method                  : {SCORING_METHOD}")
print(f"🎲 Deterministic Random Seed       : {RANDOM_SEED}")

In [ ]:
# ==============================================================================
# STAGE 4 & 5: AUTHORITATIVE FROZEN MANIFEST ENFORCEMENT & CLUSTER AUDIT
# ==============================================================================
import zlib
import base64
import hashlib
import io
import pandas as pd

def load_and_verify_manifest():
    search_paths = [
        Path("main/results/decisive_validation_manifest.csv"),
        Path("results/decisive_validation_manifest.csv"),
        Path("./decisive_validation_manifest.csv"),
        Path("/kaggle/input/indorad-vqa-research/main/results/decisive_validation_manifest.csv"),
        Path("/kaggle/input/indorad-vqa-research/results/decisive_validation_manifest.csv"),
    ]
    df_manifest = None
    loaded_source = None
    
    for sp in search_paths:
        if sp.exists():
            content = sp.read_bytes()
            if hashlib.sha256(content).hexdigest() == AUTHORITATIVE_MANIFEST_SHA256:
                df_manifest = pd.read_csv(io.BytesIO(content))
                loaded_source = str(sp)
                break
                
    if df_manifest is None:
        print("ℹ️ Loading authenticated embedded manifest...")
        raw = zlib.decompress(base64.b64decode(EMBEDDED_MANIFEST_B64.encode("ascii")))
        assert hashlib.sha256(raw).hexdigest() == AUTHORITATIVE_MANIFEST_SHA256
        df_manifest = pd.read_csv(io.BytesIO(raw))
        loaded_source = "EMBEDDED_AUTHENTICATED_MANIFEST"
        
        # Save a copy to disk for full transparency
        manifest_copy = RESULTS_ROOT / "decisive_validation_manifest.csv"
        manifest_copy.write_bytes(raw)
        
    # Invariant assertions
    assert len(df_manifest) == 150, f"Expected 150 rows, got {len(df_manifest)}"
    test_df = df_manifest[df_manifest["split"] == "test"]
    dev_df = df_manifest[df_manifest["split"] == "dev"]
    assert len(test_df) == 92, f"Expected 92 test QA pairs, got {len(test_df)}"
    assert len(dev_df) == 58, f"Expected 58 dev QA pairs, got {len(dev_df)}"
    
    test_clusters = set(test_df["canonical_image_id"].unique())
    dev_clusters = set(dev_df["canonical_image_id"].unique())
    assert len(test_clusters) == 35, f"Expected 35 test clusters, got {len(test_clusters)}"
    assert len(dev_clusters) == 23, f"Expected 23 dev clusters, got {len(dev_clusters)}"
    overlap = test_clusters.intersection(dev_clusters)
    assert len(overlap) == 0, f"Cross-split cluster leakage detected! Overlapping: {overlap}"
    
    print(f"✅ Authoritative Manifest 100% Verified ({loaded_source})")
    print(f"   SHA256: {AUTHORITATIVE_MANIFEST_SHA256}")
    print(f"   Rows  : 150 QA pairs (92 Test / 58 Dev)")
    print(f"   Images: 58 Unique Clusters (35 Test / 23 Dev | ZERO LEAKAGE)")
    return df_manifest

df_manifest = load_and_verify_manifest()

In [ ]:
# ==============================================================================
# STAGE 6: MULTI-STRATEGY IMAGE RESOLVER & PIXEL SHA-256 VERIFICATION
# ==============================================================================
from PIL import Image
import hashlib
import io
import zipfile

def compute_rgb_pixel_sha256(pil_img):
    rgb = pil_img.convert("RGB")
    return hashlib.sha256(rgb.tobytes()).hexdigest()

def resolve_all_images(df):
    cache_dir = RESULTS_ROOT / "verified_images"
    cache_dir.mkdir(parents=True, exist_ok=True)
    
    search_dirs = [
        cache_dir,
        Path("main/data/images"),
        Path("data/images"),
        Path("./images")
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        # 1. Auto-extract any zip archives found in /kaggle/input
        for zf in kaggle_input.rglob("*.zip"):
            try:
                print(f"📦 Auto-extracting zip archive: {zf.name} to {cache_dir}...")
                with zipfile.ZipFile(zf, "r") as z:
                    z.extractall(cache_dir)
            except Exception as e:
                print(f"⚠️ Warning: Could not extract {zf.name}: {e}")
                
        # 2. Discover all directories containing PNG images
        for p in kaggle_input.rglob("*.png"):
            if p.parent not in search_dirs:
                search_dirs.append(p.parent)
                
    resolved_map = {}
    missing_rows = []
    
    print("🔍 Cryptographically verifying pixel SHA-256 for all 150 images...")
    for _, row in df.iterrows():
        qa_id = row["qa_id"]
        cid = str(row["canonical_image_id"])
        expected_hash = str(row["pixel_sha256"]).strip().lower()
        expected_w = int(row["image_width"])
        expected_h = int(row["image_height"])
        
        found = None
        for sdir in search_dirs:
            for cand_name in [f"{cid}.png", f"img_{int(qa_id):05d}.png", f"{qa_id}.png", f"{row.get('image_name', '')}"]:
                if not cand_name:
                    continue
                p = sdir / cand_name
                if p.exists():
                    try:
                        with Image.open(p) as img:
                            if img.size == (expected_w, expected_h):
                                if compute_rgb_pixel_sha256(img) == expected_hash:
                                    found = p
                                    break
                    except Exception:
                        pass
            if found is not None:
                break
                
        if found is not None:
            resolved_map[qa_id] = found
        else:
            missing_rows.append(row)
            
    if missing_rows:
        print(f"⚠️ {len(missing_rows)} images missing from local search dirs. Downloading via Parquet...")
        import pyarrow.parquet as pq
        from huggingface_hub import hf_hub_download
        splits_needed = set(r["split"] for r in missing_rows)
        for sname in splits_needed:
            fname = f"data/{sname}-00000-of-00001.parquet"
            local_pq = hf_hub_download(repo_id="Lab-IS/IndoRad-VQA", filename=fname, repo_type="dataset")
            table = pq.read_table(local_pq)
            df_pq = table.to_pandas()
            for r in missing_rows:
                if r["split"] != sname:
                    continue
                qid = r["qa_id"]
                cid = str(r["canonical_image_id"])
                exp_h = str(r["pixel_sha256"]).strip().lower()
                match = df_pq[df_pq["qid"] == qid]
                if len(match) == 0:
                    continue
                raw_bytes = match.iloc[0]["image"].get("bytes", None)
                if raw_bytes:
                    with Image.open(io.BytesIO(raw_bytes)) as pil_img:
                        if compute_rgb_pixel_sha256(pil_img) == exp_h:
                            sp = cache_dir / f"{cid}.png"
                            pil_img.save(sp)
                            resolved_map[qid] = sp
                            
    unresolved = [r["qa_id"] for _, r in df.iterrows() if r["qa_id"] not in resolved_map]
    assert len(unresolved) == 0, f"Critical failure: {len(unresolved)} images unresolved!"
    print(f"✅ Cryptographic Image Verification: 100% PASS (150/150 images verified).")
    return resolved_map

image_map = resolve_all_images(df_manifest)

In [ ]:
# ==============================================================================
# STAGE 7 & 8: CANDIDATE TOKENIZATION AUDIT & SCORING IMPLEMENTATION
# ==============================================================================
import torch
import torch.nn.functional as F
from qwen_vl_utils import process_vision_info

def extract_target_loglikelihood_v3(model, processor, image_path, prompt_text, candidate_strings, target_device):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path), "min_pixels": 256 * 28 * 28, "max_pixels": 512 * 512},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    prompt_str = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    prompt_inputs = processor(
        text=[prompt_str],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(target_device)
    prompt_len = prompt_inputs.input_ids.shape[1]
    
    cand_stats = {}
    for cand in candidate_strings:
        full_text = prompt_str + cand
        inputs = processor(
            text=[full_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(target_device)
        
        # Symmetrical boundary prefix assertion
        assert torch.equal(inputs.input_ids[0, :prompt_len], prompt_inputs.input_ids[0]),             f"Prefix boundary mismatch! prompt_len={prompt_len}, inputs_len={inputs.input_ids.shape[1]}"
            
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0]
            
        target_ids = inputs.input_ids[0, prompt_len:].tolist()
        if len(target_ids) == 0:
            cand_stats[cand] = {"raw_sum": -999.0, "mean_lp": -999.0, "target_ids": [], "token_lps": []}
            del inputs, outputs
            continue
            
        target_logits = logits[prompt_len - 1 : prompt_len - 1 + len(target_ids)]
        log_probs = F.log_softmax(target_logits, dim=-1)
        token_lps = [log_probs[i, target_ids[i]].item() for i in range(len(target_ids))]
        raw_sum = float(sum(token_lps))
        mean_lp = float(raw_sum / len(target_ids))
        
        cand_stats[cand] = {
            "raw_sum": raw_sum,
            "mean_lp": mean_lp,
            "target_ids": target_ids,
            "token_lps": token_lps
        }
        del inputs, outputs
        
    del prompt_inputs
    
    c1, c2 = candidate_strings[0], candidate_strings[1]
    # Primary rev03 length-normalized relative probability
    s1_norm = cand_stats[c1]["mean_lp"]
    s2_norm = cand_stats[c2]["mean_lp"]
    max_norm = max(s1_norm, s2_norm)
    exp_norm1 = np.exp(s1_norm - max_norm)
    exp_norm2 = np.exp(s2_norm - max_norm)
    p1_norm = float(exp_norm1 / (exp_norm1 + exp_norm2))
    p2_norm = float(1.0 - p1_norm)
    
    # Raw sequence logprob (for ablation audit)
    s1_raw = cand_stats[c1]["raw_sum"]
    s2_raw = cand_stats[c2]["raw_sum"]
    max_raw = max(s1_raw, s2_raw)
    exp_raw1 = np.exp(s1_raw - max_raw)
    exp_raw2 = np.exp(s2_raw - max_raw)
    p1_raw = float(exp_raw1 / (exp_raw1 + exp_raw2))
    p2_raw = float(1.0 - p1_raw)
    
    return cand_stats, (p1_norm, p2_norm), (p1_raw, p2_raw)

def score_binary_candidates_v3(model, processor, image_path, prompt_text, candidate_strings, target_device):
    cand_stats, (p_aff, p_neg), (p_aff_raw, p_neg_raw) = extract_target_loglikelihood_v3(
        model, processor, image_path, prompt_text, candidate_strings, target_device
    )
    if p_aff >= 0.5:
        pred_label = "YES"
        confidence = p_aff
    else:
        pred_label = "NO"
        confidence = p_neg
    return pred_label, confidence, p_aff, p_neg, cand_stats, p_aff_raw, p_neg_raw

print("✅ Candidate Scoring Engine (rev03) compiled successfully.")

In [ ]:
# ==============================================================================
# STAGE 16 & 17: DISCRETE AURC, CLUSTER BOOTSTRAP & PERMUTATION TEST
# ==============================================================================

def compute_discrete_risk_coverage(scores, correctness):
    scores = np.asarray(scores)
    correctness = np.asarray(correctness, dtype=float)
    n = len(scores)
    order = np.argsort(-scores, kind="mergesort")
    sorted_corr = correctness[order]
    k = np.arange(1, n + 1)
    cov = k / n
    acc = np.cumsum(sorted_corr) / k
    risk = 1.0 - acc
    aurc_mean = float(np.mean(risk))
    return cov, risk, aurc_mean

def get_risk_at_realized_coverage(cov_arr, risk_arr, target_cov):
    idx = int((np.abs(cov_arr - target_cov)).argmin())
    return float(risk_arr[idx]), float(cov_arr[idx]), int(idx + 1)

def paired_cluster_permutation_test(df, score_col_a, score_col_b,
                                    corr_col="correct_id", cluster_col="canonical_image_id",
                                    n_perm=1000, seed=42):
    rng = np.random.default_rng(seed)
    corr = df[corr_col].to_numpy(dtype=float)
    scores_a = df[score_col_a].to_numpy(dtype=float)
    scores_b = df[score_col_b].to_numpy(dtype=float)
    clusters = df[cluster_col].to_numpy()
    
    unique_clusters = np.unique(clusters)
    n_clusters = len(unique_clusters)
    cluster_indices = [np.where(clusters == c)[0] for c in unique_clusters]
    
    _, _, aurc_a = compute_discrete_risk_coverage(scores_a, corr)
    _, _, aurc_b = compute_discrete_risk_coverage(scores_b, corr)
    t_obs = float(aurc_a - aurc_b)
    abs_t_obs = abs(t_obs)
    
    count_extreme = 0
    for _ in range(n_perm):
        flips = rng.choice([-1, 1], size=n_clusters)
        perm_a = scores_a.copy()
        perm_b = scores_b.copy()
        for idxs, f in zip(cluster_indices, flips):
            if f == -1:
                perm_a[idxs] = scores_b[idxs]
                perm_b[idxs] = scores_a[idxs]
                
        _, _, perm_aurc_a = compute_discrete_risk_coverage(perm_a, corr)
        _, _, perm_aurc_b = compute_discrete_risk_coverage(perm_b, corr)
        t_perm = perm_aurc_a - perm_aurc_b
        if abs(t_perm) >= abs_t_obs - 1e-12:
            count_extreme += 1
            
    p_val = float((1.0 + count_extreme) / (n_perm + 1.0))
    return aurc_a, aurc_b, t_obs, p_val

def run_paired_cluster_bootstrap(df, score_cols, corr_col="correct_id",
                                cluster_col="canonical_image_id", n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    unique_clusters = df[cluster_col].unique()
    n_clusters = len(unique_clusters)
    cluster_dfs = {c: df[df[cluster_col] == c] for c in unique_clusters}
    
    boot_aurcs = {col: [] for col in score_cols}
    boot_diffs = {col: [] for col in score_cols if col != "score_m1"}
    
    for _ in range(n_boot):
        sampled_clusters = rng.choice(unique_clusters, size=n_clusters, replace=True)
        boot_df = pd.concat([cluster_dfs[c] for c in sampled_clusters], ignore_index=True)
        corr = boot_df[corr_col].values
        
        aurc_m1 = None
        for col in score_cols:
            _, _, aurc = compute_discrete_risk_coverage(boot_df[col].values, corr)
            boot_aurcs[col].append(aurc)
            if col == "score_m1":
                aurc_m1 = aurc
                
        for col in boot_diffs.keys():
            boot_diffs[col].append(boot_aurcs[col][-1] - aurc_m1)
            
    summary = {}
    for col in score_cols:
        vals = np.array(boot_aurcs[col])
        summary[col] = {
            "mean": float(np.mean(vals)),
            "std": float(np.std(vals)),
            "ci_95_lower": float(np.percentile(vals, 2.5)),
            "ci_95_upper": float(np.percentile(vals, 97.5))
        }
    diff_summary = {}
    for col in boot_diffs.keys():
        diff_vals = np.array(boot_diffs[col])
        diff_summary[col] = {
            "mean_diff": float(np.mean(diff_vals)),
            "ci_95_lower": float(np.percentile(diff_vals, 2.5)),
            "ci_95_upper": float(np.percentile(diff_vals, 97.5))
        }
    return summary, diff_summary

def apply_holm_bonferroni(p_dict):
    sorted_items = sorted(p_dict.items(), key=lambda x: x[1])
    m = len(sorted_items)
    adj_p = {}
    running_max = 0.0
    for rank, (k, p) in enumerate(sorted_items, 1):
        mult = m - rank + 1
        val = min(1.0, p * mult)
        running_max = max(running_max, val)
        adj_p[k] = min(1.0, running_max)
    return adj_p

print("✅ Statistical Testing Engine (Exact Permutation + Holm) compiled successfully.")

In [ ]:
# ==============================================================================
# STAGE 9-12: MODEL PIPELINE EXECUTOR & CHECKPOINT GOVERNANCE
# ==============================================================================
import gc
import time
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoConfig, BitsAndBytesConfig
import transformers

def validate_checkpoint_compatibility(df_ckpt, model_name, expected_manifest_sha256, expected_qa_ids):
    required_cols = [
        "qa_id", "canonical_image_id", "split", "reference_answer",
        "pred_en", "conf_en", "correct_en", "p_en_yes", "p_en_no",
        "pred_id", "conf_id", "correct_id", "p_orig_ya", "p_orig_tidak",
        "p_blank_ya", "p_blank_tidak", "p_blank_pred", "delta_p_dir", "delta_p_abs",
        "score_m1", "score_m2", "score_m_dir", "score_m_norm", "score_m3_old",
        "scoring_method", "revision_id", "manifest_sha256"
    ]
    for c in required_cols:
        if c not in df_ckpt.columns:
            return False, f"Missing required column: {c}"
    rev_vals = df_ckpt["revision_id"].dropna().unique()
    if len(rev_vals) == 0 or rev_vals[0] != REVISION_ID:
        return False, f"Incompatible revision ID! Expected {REVISION_ID}, got {rev_vals}"
    sha_vals = df_ckpt["manifest_sha256"].dropna().unique()
    if len(sha_vals) == 0 or sha_vals[0] != expected_manifest_sha256:
        return False, f"Manifest SHA256 mismatch! Expected {expected_manifest_sha256}, got {sha_vals}"
    model_vals = df_ckpt["model_id"].dropna().unique()
    if len(model_vals) == 0 or model_vals[0] != model_name:
        return False, f"Model ID mismatch! Expected {model_name}, got {model_vals}"
    ckpt_qa_ids = set(df_ckpt["qa_id"].tolist())
    if not ckpt_qa_ids.issubset(expected_qa_ids):
        return False, f"Found unexpected QA IDs in checkpoint"
    return True, "Valid rev03 checkpoint"

def execute_model_evaluation_v3(model_name, df_eval, image_map, results_root, perturb_dir):
    safe_name = model_name.replace("/", "_").replace("-", "_")
    model_out_dir = results_root / safe_name
    model_out_dir.mkdir(parents=True, exist_ok=True)
    perturb_dir.mkdir(parents=True, exist_ok=True)
    
    checkpoint_csv = model_out_dir / f"live_validation_checkpoint_{safe_name}.csv"
    expected_qa_ids = set(df_eval["qa_id"].tolist())
    completed_qa_ids = set()
    results = []
    
    if checkpoint_csv.exists():
        try:
            df_old = pd.read_csv(checkpoint_csv)
            is_valid, msg = validate_checkpoint_compatibility(
                df_old, model_name, AUTHORITATIVE_MANIFEST_SHA256, expected_qa_ids
            )
            if is_valid:
                completed_qa_ids = set(df_old["qa_id"].tolist())
                results = df_old.to_dict(orient="records")
                print(f"🔄 Resuming from valid {REVISION_ID} checkpoint: {len(completed_qa_ids)}/{len(df_eval)} completed.")
            else:
                print(f"⚠️ Incompatible checkpoint detected ({msg}). Backing up...")
                backup_p = model_out_dir / f"incompatible_checkpoint_backup_{int(time.time())}.csv"
                checkpoint_csv.rename(backup_p)
        except Exception as e:
            print(f"⚠️ Checkpoint read error: {e}")
            
    if len(completed_qa_ids) >= len(df_eval):
        print(f"✅ Model [{model_name}] already 100% completed in {REVISION_ID}.")
        return pd.DataFrame(results), model_out_dir

    print(f"\n📦 Loading Model: {model_name}...")
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    arch_list = getattr(config, "architectures", [])
    model_class = None
    for arch_name in arch_list:
        if hasattr(transformers, arch_name):
            model_class = getattr(transformers, arch_name)
            break
    if model_class is None:
        try:
            from transformers import AutoModelForImageTextToText
            model_class = AutoModelForImageTextToText
        except ImportError:
            from transformers import AutoModelForVision2Seq
            model_class = AutoModelForVision2Seq
            
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True
    )
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = model_class.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        trust_remote_code=True
    )
    model.eval()
    target_dev = next(model.parameters()).device
    print(f"✅ Loaded onto: {target_dev}")

    for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc=f"Evaluating {model_name.split('/')[-1]}"):
        qa_id = row["qa_id"]
        if qa_id in completed_qa_ids:
            continue
            
        cid = str(row["canonical_image_id"])
        img_id = str(row.get("image_id", cid))
        split_val = str(row.get("split", "test"))
        trans_stat = str(row.get("translation_status", "CLEAN"))
        q_en = str(row["question_en"])
        q_id = str(row["question_id"])
        ref_ans = str(row["reference_answer"]).strip().upper()
        orig_img_path = image_map[qa_id]
        
        # P1 Black Blank
        blank_img_path = perturb_dir / f"blank_{cid}.png"
        if not blank_img_path.exists():
            with Image.open(orig_img_path) as orig_pil:
                blank_pil = Image.new("RGB", orig_pil.size, color=(0, 0, 0))
                blank_pil.save(blank_img_path)
                
        # RQ1 Bilingual Scoring on P0
        prompt_en = PROMPT_EN_TEMPLATE.format(q=q_en)
        pred_en, conf_en, p_en_yes, p_en_no, _, p_en_yes_raw, p_en_no_raw = score_binary_candidates_v3(
            model, processor, orig_img_path, prompt_en, CANDIDATES_EN, target_dev
        )
        corr_en = int(pred_en == ref_ans)
        
        prompt_id = PROMPT_ID_TEMPLATE.format(q=q_id)
        pred_id, conf_id, p_orig_ya, p_orig_tidak, _, p_orig_ya_raw, p_orig_tidak_raw = score_binary_candidates_v3(
            model, processor, orig_img_path, prompt_id, CANDIDATES_ID, target_dev
        )
        corr_id = int(pred_id == ref_ans)
        flip = int(pred_en != pred_id)
        joint_corr = int(corr_en and corr_id)
        
        # RQ2 Visual Intervention Scoring on P1
        _, _, p_blank_ya, p_blank_tidak, _, p_blank_ya_raw, p_blank_tidak_raw = score_binary_candidates_v3(
            model, processor, blank_img_path, prompt_id, CANDIDATES_ID, target_dev
        )
        p_blank_pred = p_blank_ya if pred_id == "YES" else p_blank_tidak
        delta_p_dir = conf_id - p_blank_pred
        delta_p_abs = abs(conf_id - p_blank_pred)
        
        # RQ3 5 Selective Prediction Estimators
        score_m1 = conf_id
        score_m2 = max(0.0, delta_p_dir)
        score_m_dir = conf_id * max(0.0, delta_p_dir)
        score_m_norm = conf_id * (1.0 + delta_p_dir) / 2.0
        score_m3_old = conf_id * (1.0 + delta_p_abs) / 2.0
        
        results.append({
            "qa_id": qa_id,
            "canonical_image_id": cid,
            "image_id": img_id,
            "split": split_val,
            "translation_status": trans_stat,
            "is_yes_no": bool(row.get("is_yes_no", True)),
            "model_id": model_name,
            "device": str(target_dev),
            "question_en": q_en,
            "question_id": q_id,
            "reference_answer": ref_ans,
            "pred_en": pred_en,
            "conf_en": conf_en,
            "correct_en": corr_en,
            "p_en_yes": p_en_yes,
            "p_en_no": p_en_no,
            "pred_id": pred_id,
            "conf_id": conf_id,
            "correct_id": corr_id,
            "p_orig_ya": p_orig_ya,
            "p_orig_tidak": p_orig_tidak,
            "flip": flip,
            "joint_correct": joint_corr,
            "p_en_yes_raw": p_en_yes_raw,
            "p_en_no_raw": p_en_no_raw,
            "p_orig_ya_raw": p_orig_ya_raw,
            "p_orig_tidak_raw": p_orig_tidak_raw,
            "p_blank_ya": p_blank_ya,
            "p_blank_tidak": p_blank_tidak,
            "p_blank_pred": p_blank_pred,
            "delta_p_dir": delta_p_dir,
            "delta_p_abs": delta_p_abs,
            "score_m1": score_m1,
            "score_m2": score_m2,
            "score_m_dir": score_m_dir,
            "score_m_norm": score_m_norm,
            "score_m3_old": score_m3_old,
            "blank_method": "PIL.Image.new('RGB', size, (0,0,0))",
            "scoring_method": SCORING_METHOD,
            "revision_id": REVISION_ID,
            "manifest_sha256": AUTHORITATIVE_MANIFEST_SHA256,
            "timestamp": datetime.now(timezone.utc).isoformat()
        })
        completed_qa_ids.add(qa_id)
        if len(results) % 5 == 0 or len(results) == len(df_eval):
            pd.DataFrame(results).to_csv(checkpoint_csv, index=False)

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    df_res = pd.DataFrame(results)
    df_res.to_csv(checkpoint_csv, index=False)
    print(f"💾 Checkpoint saved: {checkpoint_csv.name} ({len(df_res)} rows)")
    return df_res, model_out_dir

In [ ]:
# ==============================================================================
# STAGE 11: INFERENCE RUN — MODEL 1 (QWEN2.5-VL-7B-INSTRUCT)
# ==============================================================================
print("🚀 Launching Model 1: Qwen/Qwen2.5-VL-7B-Instruct...")
df_qwen, qwen_dir = execute_model_evaluation_v3(
    DEFAULT_MODEL_1, df_manifest, image_map, RESULTS_ROOT, PERTURB_DIR
)
print("✅ Model 1 evaluation complete.")

In [ ]:
# ==============================================================================
# STAGE 12: INFERENCE RUN — MODEL 2 (HUATUOGPT-VISION-7B-QWEN2.5VL)
# ==============================================================================
print("🚀 Launching Model 2: FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL...")
df_huatuo, huatuo_dir = execute_model_evaluation_v3(
    DEFAULT_MODEL_2, df_manifest, image_map, RESULTS_ROOT, PERTURB_DIR
)
print("✅ Model 2 evaluation complete.")

In [ ]:
# ==============================================================================
# STAGE 13 & 14: RQ1 BILINGUAL EVALUATION & PREDICTION DISTRIBUTION AUDIT
# ==============================================================================
def summarize_rq1(df, model_title):
    test_df = df[df["split"] == "test"]
    n = len(test_df)
    acc_en = test_df["correct_en"].mean() * 100.0
    acc_id = test_df["correct_id"].mean() * 100.0
    gap = acc_en - acc_id
    flip = test_df["flip"].mean() * 100.0
    joint = test_df["joint_correct"].mean() * 100.0
    
    gt_dist = test_df["reference_answer"].value_counts().to_dict()
    pred_en_dist = test_df["pred_en"].value_counts().to_dict()
    pred_id_dist = test_df["pred_id"].value_counts().to_dict()
    
    print(f"--- RQ1 BILINGUAL METRICS: {model_title} (Primary Test N={n}) ---")
    print(f"Ground Truth Distribution : {gt_dist}")
    print(f"English Accuracy          : {acc_en:.2f}% | Predictions: {pred_en_dist}")
    print(f"Indonesian Accuracy       : {acc_id:.2f}% | Predictions: {pred_id_dist}")
    print(f"Cross-Lingual Gap (EN-ID) : {gap:+.2f} pp")
    print(f"Prediction Flip Rate      : {flip:.2f}%")
    print(f"Joint Bilingual Correct   : {joint:.2f}%\n")
    return {
        "model": model_title, "acc_en": acc_en, "acc_id": acc_id,
        "gap": gap, "flip_rate": flip, "joint_acc": joint,
        "pred_en_yes": pred_en_dist.get("YES", 0), "pred_en_no": pred_en_dist.get("NO", 0),
        "pred_id_yes": pred_id_dist.get("YES", 0), "pred_id_no": pred_id_dist.get("NO", 0),
    }

rq1_qwen = summarize_rq1(df_qwen, "Qwen2.5-VL-7B")
rq1_huatuo = summarize_rq1(df_huatuo, "HuatuoGPT-Vision-7B")

df_rq1 = pd.DataFrame([rq1_qwen, rq1_huatuo])
rq1_csv = RESULTS_ROOT / "rq1_primary_test_only.csv"
df_rq1.to_csv(rq1_csv, index=False)
print(f"💾 Saved RQ1 Primary Metrics: {rq1_csv.name}")

In [ ]:
# ==============================================================================
# STAGE 15: RQ2 VISUAL INTERVENTION (P0 ORIGINAL VS P1 BLANK)
# ==============================================================================
def summarize_rq2(df, model_title):
    test_df = df[df["split"] == "test"]
    n = len(test_df)
    dp = test_df["delta_p_dir"].values
    mean_dp = float(np.mean(dp))
    median_dp = float(np.median(dp))
    pos = float(np.mean(dp > 0)) * 100.0
    neg = float(np.mean(dp < 0)) * 100.0
    zero = float(np.mean(dp == 0)) * 100.0
    
    print(f"--- RQ2 VISUAL INTERVENTION: {model_title} (Primary Test N={n}) ---")
    print(f"Mean Delta P_dir          : {mean_dp:+.4f}")
    print(f"Median Delta P_dir        : {median_dp:+.4f}")
    print(f"Positive Sensitivity (>0) : {pos:.2f}% (Model degraded when image blinded)")
    print(f"Negative Sensitivity (<0) : {neg:.2f}% (Model gained confidence without image)")
    print(f"Zero Sensitivity (==0)    : {zero:.2f}%\n")
    return {
        "model": model_title, "mean_delta_p": mean_dp, "median_delta_p": median_dp,
        "prop_positive": pos, "prop_negative": neg, "prop_zero": zero
    }

rq2_qwen = summarize_rq2(df_qwen, "Qwen2.5-VL-7B")
rq2_huatuo = summarize_rq2(df_huatuo, "HuatuoGPT-Vision-7B")

df_rq2 = pd.DataFrame([rq2_qwen, rq2_huatuo])
rq2_csv = RESULTS_ROOT / "rq2_visual_intervention_test_only.csv"
df_rq2.to_csv(rq2_csv, index=False)
print(f"💾 Saved RQ2 Visual Intervention Metrics: {rq2_csv.name}")

In [ ]:
# ==============================================================================
# STAGE 16-18: RQ3 SELECTIVE PREDICTION & EXACT STATISTICAL TESTING (TEST-ONLY)
# ==============================================================================
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

def run_selective_prediction_audit(df, model_title, out_dir):
    test_df = df[df["split"] == "test"]
    n_samples = len(test_df)
    n_clusters = test_df["canonical_image_id"].nunique()
    score_cols = ["score_m1", "score_m2", "score_m_dir", "score_m_norm", "score_m3_old"]
    
    print(f"\n🔬 Running Selective Prediction & Permutation Test for {model_title} (Test-Only N={n_samples})...")
    boot_sum, boot_diff = run_paired_cluster_bootstrap(test_df, score_cols, n_boot=1000, seed=42)
    
    rows = []
    raw_p_dict = {}
    for col in score_cols:
        cov, risk, aurc = compute_discrete_risk_coverage(test_df[col].values, test_df["correct_id"].values)
        r_100, _, _ = get_risk_at_realized_coverage(cov, risk, 1.0)
        r_90, _, _ = get_risk_at_realized_coverage(cov, risk, 0.9)
        r_80, _, _ = get_risk_at_realized_coverage(cov, risk, 0.8)
        r_70, _, _ = get_risk_at_realized_coverage(cov, risk, 0.7)
        r_50, _, _ = get_risk_at_realized_coverage(cov, risk, 0.5)
        
        if col != "score_m1":
            _, _, t_obs, p_val = paired_cluster_permutation_test(
                test_df, col, "score_m1", n_perm=1000, seed=42
            )
            raw_p_dict[col] = p_val
        else:
            t_obs = 0.0
            p_val = 1.0
            
        rows.append({
            "model": model_title,
            "subset": "test_only",
            "estimator": col,
            "aurc_discrete_mean": aurc,
            "ci_95_lower": boot_sum[col]["ci_95_lower"],
            "ci_95_upper": boot_sum[col]["ci_95_upper"],
            "delta_aurc_vs_m1": (aurc - rows[0]["aurc_discrete_mean"]) if col != "score_m1" else 0.0,
            "delta_ci_95_lower": boot_diff[col]["ci_95_lower"] if col != "score_m1" else 0.0,
            "delta_ci_95_upper": boot_diff[col]["ci_95_upper"] if col != "score_m1" else 0.0,
            "perm_p_value_raw": p_val,
            "risk_cov_100": r_100,
            "risk_cov_90": r_90,
            "risk_cov_80": r_80,
            "risk_cov_70": r_70,
            "risk_cov_50": r_50,
        })
        
    adj_p = apply_holm_bonferroni(raw_p_dict)
    for r in rows:
        est = r["estimator"]
        r["perm_p_value_holm"] = adj_p.get(est, 1.0 if est == "score_m1" else 1.0)
        
    df_res = pd.DataFrame(rows)
    safe_name = model_title.replace("/", "_").replace("-", "_")
    csv_path = out_dir / f"estimator_comparison_{safe_name}.csv"
    df_res.to_csv(csv_path, index=False)
    
    # Plot Risk-Coverage Curve
    fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=300)
    color_map = {
        "score_m1": ("#1f77b4", "M1: Confidence-Only (p_orig)", "-"),
        "score_m2": ("#ff7f0e", "M2: Visual-Only max(0, Delta)", "--"),
        "score_m_dir": ("#2ca02c", "M_dir: p_orig * max(0, Delta)", "-"),
        "score_m_norm": ("#9467bd", "M_norm: p_orig * (1 + Delta)/2", "-."),
        "score_m3_old": ("#d62728", "M3_old: Historical Unsigned", ":")
    }
    for col in score_cols:
        cov, risk, aurc = compute_discrete_risk_coverage(test_df[col].values, test_df["correct_id"].values)
        color, label, linestyle = color_map[col]
        p_adj_val = df_res.loc[df_res["estimator"] == col, "perm_p_value_holm"].values[0]
        p_str = f" | p_adj={p_adj_val:.3f}" if col != "score_m1" else " [Base]"
        ax.plot(cov, risk, label=f"{label} (AURC={aurc:.4f}{p_str})", color=color, linestyle=linestyle, linewidth=2.0)
        
    ax.set_title(f"Selective Risk-Coverage Curves: {model_title}\nPrimary Test Set (N={n_samples}, Clusters={n_clusters})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Coverage (Fraction of Accepted Predictions)", fontsize=10)
    ax.set_ylabel("Selective Empirical Risk (1 - Accuracy)", fontsize=10)
    ax.set_xlim(0.0, 1.02)
    ax.set_ylim(-0.02, 0.65)
    ax.legend(loc="upper right", frameon=True, fontsize=9)
    fig.tight_layout()
    plot_p = out_dir / f"risk_coverage_curve_{safe_name}_test.png"
    fig.savefig(plot_p)
    plt.close(fig)
    
    print(f"💾 Exported: {csv_path.name} & {plot_p.name}")
    return df_res

est_qwen = run_selective_prediction_audit(df_qwen, "Qwen2.5-VL-7B", qwen_dir)
est_huatuo = run_selective_prediction_audit(df_huatuo, "HuatuoGPT-Vision-7B", huatuo_dir)

In [ ]:
# ==============================================================================
# STAGE 19: FULL BENCHMARK SENSITIVITY AUDIT (N=150 SAMPLES)
# ==============================================================================
def run_full_benchmark_sensitivity(df, model_title, out_dir):
    n_samples = len(df)
    score_cols = ["score_m1", "score_m2", "score_m_dir", "score_m_norm", "score_m3_old"]
    boot_sum, boot_diff = run_paired_cluster_bootstrap(df, score_cols, n_boot=1000, seed=42)
    rows = []
    raw_p = {}
    for col in score_cols:
        cov, risk, aurc = compute_discrete_risk_coverage(df[col].values, df["correct_id"].values)
        if col != "score_m1":
            _, _, t_obs, p_val = paired_cluster_permutation_test(df, col, "score_m1", n_perm=1000, seed=42)
            raw_p[col] = p_val
        else:
            p_val = 1.0
        rows.append({
            "model": model_title, "subset": "full_benchmark_sensitivity", "estimator": col,
            "aurc_discrete_mean": aurc, "ci_95_lower": boot_sum[col]["ci_95_lower"],
            "ci_95_upper": boot_sum[col]["ci_95_upper"],
            "delta_aurc_vs_m1": (aurc - rows[0]["aurc_discrete_mean"]) if col != "score_m1" else 0.0,
            "perm_p_value_raw": p_val
        })
    adj = apply_holm_bonferroni(raw_p)
    for r in rows:
        est = r["estimator"]
        r["perm_p_value_holm"] = adj.get(est, 1.0 if est == "score_m1" else 1.0)
    df_sens = pd.DataFrame(rows)
    safe = model_title.replace("/", "_").replace("-", "_")
    p = out_dir / f"sensitivity_full_benchmark_{safe}.csv"
    df_sens.to_csv(p, index=False)
    print(f"💾 Saved sensitivity analysis: {p.name}")
    return df_sens

sens_qwen = run_full_benchmark_sensitivity(df_qwen, "Qwen2.5-VL-7B", qwen_dir)
sens_huatuo = run_full_benchmark_sensitivity(df_huatuo, "HuatuoGPT-Vision-7B", huatuo_dir)

In [ ]:
# ==============================================================================
# STAGE 20: CROSS-MODEL SYNTHESIS TABLE (TEST-ONLY PRIMARY)
# ==============================================================================
df_cross = pd.concat([est_qwen, est_huatuo], ignore_index=True)
cross_csv = RESULTS_ROOT / "final_cross_model_comparison_test_only.csv"
df_cross.to_csv(cross_csv, index=False)

print("=" * 75)
print("📊 FINAL CROSS-MODEL COMPARISON (PRIMARY TEST SET N=92)")
print("=" * 75)
cols_show = ["model", "estimator", "aurc_discrete_mean", "ci_95_lower", "ci_95_upper", "delta_aurc_vs_m1", "perm_p_value_holm"]
display(df_cross[cols_show])
print("=" * 75)

In [ ]:
# ==============================================================================
# STAGE 21 & 22: METADATA, SHA-256 MANIFEST & FINAL REPRODUCIBILITY ZIP
# ==============================================================================
import zipfile

# 1. Generate run_metadata.json
meta = {
    "revision_id": REVISION_ID,
    "scoring_method": SCORING_METHOD,
    "manifest_sha256": AUTHORITATIVE_MANIFEST_SHA256,
    "random_seed": RANDOM_SEED,
    "candidates_en": CANDIDATES_EN,
    "candidates_id": CANDIDATES_ID,
    "prompt_en": PROMPT_EN_TEMPLATE,
    "prompt_id": PROMPT_ID_TEMPLATE,
    "completed_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_total": len(df_manifest),
    "n_test": int((df_manifest["split"] == "test").sum()),
    "n_dev": int((df_manifest["split"] == "dev").sum()),
}
meta_p = RESULTS_ROOT / "run_metadata.json"
meta_p.write_text(json.dumps(meta, indent=2))

# 2. Build SHA-256 Manifest of all produced artifacts
artifact_rows = []
for root, dirs, files in os.walk(RESULTS_ROOT):
    for f in files:
        if f.endswith(".zip"):
            continue
        fp = Path(root) / f
        h = hashlib.sha256(fp.read_bytes()).hexdigest()
        artifact_rows.append({"file_name": fp.name, "relative_path": str(fp.relative_to(RESULTS_ROOT)), "sha256": h})

df_artifacts = pd.DataFrame(artifact_rows)
art_manifest_p = RESULTS_ROOT / "artifact_sha256_manifest.csv"
df_artifacts.to_csv(art_manifest_p, index=False)

# 3. Create Final Reproducibility Zip
zip_name = "decisive_validation_rev03_results_final.zip"
zip_path = RESULTS_ROOT / zip_name
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(RESULTS_ROOT):
        for file in files:
            p = Path(root) / file
            if p != zip_path:
                zipf.write(p, p.relative_to(RESULTS_ROOT))

# Also copy zip to working directory root for 1-click download from Kaggle output panel
root_zip = Path(zip_name)
if root_zip.resolve() != zip_path.resolve():
    import shutil
    shutil.copy2(zip_path, root_zip)

print("=" * 75)
print(f"📦 REPRODUCIBILITY ARCHIVE COMPLETE: {zip_path.name}")
print(f"   Size: {zip_path.stat().st_size / (1024*1024):.2f} MB")
print("=" * 75)
print("🔗 Download links for Kaggle environment:")
display(FileLink(str(root_zip)))
display(FileLink(str(zip_path)))